In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Rohini, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,194.44,271.65,13.79,44.81,35.25,1.36,23.21,0.15,8.05,84.87,1.38,259.45,61.00,988.46,12.34,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,196.67,280.42,18.04,38.61,35.22,1.60,21.11,0.15,26.16,87.36,1.46,237.36,52.35,987.33,12.44,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,255.46,362.33,58.68,41.65,70.01,2.57,15.96,0.21,13.41,88.98,1.14,230.79,60.92,987.70,14.16,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,278.57,372.43,26.22,64.15,55.44,2.41,16.11,0.27,13.00,91.27,1.46,118.81,57.43,986.27,13.99,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,193.29,286.38,17.15,53.60,42.55,1.63,15.49,0.09,9.17,83.00,1.32,110.67,61.66,986.19,14.46,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,415.43,599.96,44.05,43.74,59.26,1.50,61.08,0.75,14.40,63.15,1.22,214.36,74.52,988.16,19.94,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,365.52,559.58,24.81,60.89,52.73,1.74,54.24,0.73,14.38,64.52,1.19,221.57,73.41,989.71,19.78,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,274.15,458.75,22.13,57.89,48.78,1.52,55.10,0.59,47.35,64.46,1.24,210.68,71.64,989.60,19.34,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,290.91,467.38,25.11,51.55,47.93,1.56,57.16,0.58,20.68,64.96,1.20,206.70,71.12,990.46,19.06,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 19)
          From Date           To Date    PM2.5    PM10     NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  194.440  271.65  13.79  44.81  35.250   
1  02-01-2025 00:00  03-01-2025 00:00  196.670  280.42  18.04  38.61  35.220   
2  03-01-2025 00:00  04-01-2025 00:00   66.125  362.33   8.02  41.65  23.275   
3  04-01-2025 00:00  05-01-2025 00:00   66.125  372.43  26.22  64.15  55.440   
4  05-01-2025 00:00  06-01-2025 00:00  193.290  286.38  17.15  53.60  42.550   

     CO  Ozone  Benzene  Toluene     RH    WS      WD     SR      BP     AT  \
0  1.36  23.21     0.15    8.050  84.87  1.38  259.45  61.00  988.46  12.34   
1  1.60  21.11     0.15    5.985  87.36  1.46  237.36  52.35  987.33  12.44   
2  0.79  15.96     0.21   13.410  88.98  1.14  230.79  60.92  987.70  14.16   
3  0.79  16.11     0.27   13.000  91.27  1.46  118.81  57.43  986.27  13.99   
4  1.63  15.49     0.09    9.170  83.00  1.32  110.67  61.66  986.19  14.46   

    RF  TOT-RF  
0  0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,2.311591,0.663791,0.698983,0.687695,0.791118,1.335145,-0.994092,-0.127729,0.414599,1.392806,-0.410729,1.444005,-1.305449,1.453838,-2.487439,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.355512,0.745125,1.372138,0.319983,0.788795,1.945243,-1.107503,-0.127729,-0.135851,1.576802,0.011377,1.027242,-1.476449,1.249409,-2.468253,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.215687,1.504770,-0.214925,0.500281,-0.136085,-0.113837,-1.385630,0.673710,1.843370,1.696510,-1.677047,0.903289,-1.307030,1.316346,-2.138243,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.215687,1.598439,2.667764,1.834719,2.354394,-0.113837,-1.377530,1.475150,1.734079,1.865727,0.011377,-1.209387,-1.376023,1.057643,-2.170860,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.288940,0.800399,1.231171,1.209016,1.356344,2.021505,-1.411013,-0.929169,0.713148,1.254625,-0.727309,-1.362961,-1.292401,1.043170,-2.080683,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.215687,-0.125485,-0.214925,0.624235,2.650169,1.691036,1.051091,0.206204,2.107266,-0.212168,-1.254941,0.593312,-1.038175,1.399564,-1.029258,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.215687,-0.125485,2.444434,1.641373,2.144563,2.301133,0.681694,0.206204,2.101935,-0.110933,-1.413231,0.729340,-1.060118,1.679976,-1.059957,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.215687,2.398983,2.019951,1.463448,1.838722,1.741877,0.728139,0.206204,-0.135851,-0.115367,-1.149415,0.523883,-1.095109,1.660076,-1.144378,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.215687,2.479019,2.491951,1.087433,1.772908,1.843560,0.839390,0.206204,-0.135851,-0.078420,-1.360468,0.448794,-1.105389,1.815659,-1.198100,0.0,0.0


In [10]:
df.to_excel('Rohini2025.xlsx', index=False)